# **Reddit Scraper Notebook for HealthPH+**


# **Dependencies**

In [13]:
import requests
import time
import pandas as pd
from datetime import datetime
from urllib.parse import urlparse, parse_qs
import os
import re


In [14]:
from pathlib import Path
from typing import Any
from urllib.parse import quote_plus
import hashlib
import json

try:
    from playwright.async_api import (
        async_playwright,
        TimeoutError as PlaywrightTimeoutError,
    )
except ImportError as exc:
    raise ImportError(
        "playwright is not installed. Run: pip install playwright && playwright install chromium"
    ) from exc

In [15]:
print("✅ Libraries loaded successfully")

✅ Libraries loaded successfully


## **Reddit**

### Configuration

In [16]:
# ── USER SETTINGS ──────────────────────────────────────────────────────────────
# Paste any Reddit search URL below.
# The query, sort order, and time filter are parsed automatically from the URL.
# Example filters you can append to the URL:
#   &sort=relevance | new | top | comments
#   &t=all | year | month | week | day | hour


KEYWORD = 'linalagnat'
SEARCH_URL = f"https://www.reddit.com/search/?q={KEYWORD}&sort=new&t=all"

LIMIT     = 50   # Posts per page (max 100)
MAX_PAGES = 10   # Number of pages to paginate through

CWD = Path.cwd()
ROOT_DIR = next((p for p in [CWD, *CWD.parents] if (p / '.git').exists()), CWD)
OUTPUT_FILE = str(ROOT_DIR / "data" / "raw" / "reddit" / "reddit_results.csv")   # Master data output file
# ───────────────────────────────────────────────────────────────────────────────

print(f"Search URL : {SEARCH_URL}")
print(f"Limit      : {LIMIT} posts/page")
print(f"Max pages  : {MAX_PAGES}")
print(f"Output file: {OUTPUT_FILE}")

Search URL : https://www.reddit.com/search/?q=linalagnat&sort=new&t=all
Limit      : 50 posts/page
Max pages  : 10
Output file: /Users/angelodelapaz/Documents/GitHub/healthphpersonal/data/raw/reddit/reddit_results.csv


In [17]:
## 3. Helper Functions
def extract_reddit_search_params(url):
    """
    Extract search query, sort order, and time filter from a Reddit search URL.
    
    Example URL:
        https://www.reddit.com/search/?q=sakit&sort=top&t=month
    
    Returns:
        tuple: (query, sort, time_filter)
    """
    parsed = urlparse(url)
    params = parse_qs(parsed.query)

    query = params.get("q", [None])[0]
    sort  = params.get("sort", ["relevance"])[0]
    t     = params.get("t",    ["all"])[0]

    if not query:
        raise ValueError(f"Could not extract a search query from URL: {url}")

    return query, sort, t


def get_last_reddit_fullname_from_csv(filepath):
    """
    Read the last saved Reddit post URL from CSV and return its fullname (t3_<id>).

    Returns None if the file is missing, empty, or the URL is not parseable.
    """
    if not os.path.isfile(filepath):
        return None

    try:
        df = pd.read_csv(filepath, usecols=["url"])
        if df.empty:
            return None

        last_url = df["url"].dropna().iloc[-1]
        if not isinstance(last_url, str) or not last_url:
            return None

        match = re.search(r"/comments/([a-z0-9]+)/", last_url)
        if not match:
            return None

        return f"t3_{match.group(1)}"
    except Exception as e:
        print(f"⚠️ Could not read last id from {filepath}: {e}")
        return None


def scrape_reddit_search(url, limit=25, max_pages=3):
    """
    Scrape Reddit search results from a Reddit search URL.

    Args:
        url       : A Reddit search URL.
        limit     : Posts per page (max 100).
        max_pages : Maximum number of pages to paginate through.

    Returns:
        pd.DataFrame with columns: title, selftext, created, ups, subreddit, url
    """
    query, sort, t = extract_reddit_search_params(url)
    print(f"Query: '{query}'  |  Sort: {sort}  |  Time filter: {t}\n")

    headers  = {"User-Agent": "RedditScraper/1.0 (personal research script)"}
    base_url = "https://www.reddit.com/search.json"
    results  = []
    after    = None  # pagination cursor (within this run only)

    for page in range(max_pages):
        params = {
            "q":     query,
            "limit": limit,
            "sort":  sort,
            "t":     t,
            "type":  "link",   # posts only
        }
        if after:
            params["after"] = after

        response = requests.get(base_url, params=params, headers=headers)

        if response.status_code != 200:
            print(f"❌ Error: HTTP {response.status_code}")
            break

        data  = response.json()
        posts = data.get("data", {}).get("children", [])

        if not posts:
            print("ℹ️  No more posts found.")
            break

        for post in posts:
            pd_  = post.get("data", {})
            results.append({
                "title":     pd_.get("title", ""),
                "selftext":  pd_.get("selftext", ""),
                "created":   datetime.fromtimestamp(
                                 pd_.get("created_utc", 0)
                             ).strftime("%Y-%m-%d %H:%M:%S UTC"),
                "ups":       pd_.get("ups", 0),
                "subreddit": pd_.get("subreddit", ""),
                "url":       f"https://reddit.com{pd_.get('permalink', '')}",
            })

        after = data.get("data", {}).get("after")
        print(f"✔ Page {page + 1}: fetched {len(posts)} posts  (running total: {len(results)})")

        if not after:
            print("ℹ️  Reached last page.")
            break

        time.sleep(1)   # polite delay to avoid rate limiting

    return pd.DataFrame(results)


def save_results(df, filepath=None):
    """
    Save Reddit posts to CSV without adding duplicates.
    Dedupe key: post URL.
    """
    if filepath is None:
        filepath = OUTPUT_FILE

    os.makedirs(os.path.dirname(filepath), exist_ok=True)

    if df.empty:
        print("ℹ️ No rows to save.")
        return

    if "url" not in df.columns:
        raise ValueError("save_results requires a 'url' column for deduplication.")

    file_exists = os.path.isfile(filepath)
    incoming_count = len(df)

    to_save = df.copy()
    to_save["url"] = to_save["url"].astype(str).str.strip()
    to_save = to_save[to_save["url"] != ""]

    # Remove duplicates in this scrape batch
    to_save = to_save.drop_duplicates(subset=["url"], keep="first")

    # Remove rows already present in the output file
    if file_exists and not to_save.empty:
        try:
            existing_urls = set(
                pd.read_csv(filepath, usecols=["url"])["url"]
                .dropna()
                .astype(str)
                .str.strip()
            )
            to_save = to_save[~to_save["url"].isin(existing_urls)]
        except ValueError:
            print("⚠️ Existing file has no 'url' column. Skipping cross-run dedupe.")

    new_rows = len(to_save)
    if new_rows == 0:
        skipped = incoming_count
        print(f"ℹ️ No new posts to append. (Skipped {skipped} duplicates)")
    else:
        to_save.to_csv(
            filepath,
            mode="a",
            index=False,
            encoding="utf-8-sig",
            header=not file_exists,
        )
        skipped = incoming_count - new_rows
        action = "Appended to" if file_exists else "Created"
        print(f"💾 {action} {filepath}  (+{new_rows} posts, skipped {skipped} duplicates)")

    total = pd.read_csv(filepath).shape[0] if os.path.isfile(filepath) else 0
    print(f"📊 Total rows in file: {total}")


print("✅ Functions defined")

✅ Functions defined


### Main Function

In [18]:
df = scrape_reddit_search(url=SEARCH_URL, limit=LIMIT, max_pages=MAX_PAGES)
print(f"Total posts collected: {len(df)}")


Query: 'linalagnat'  |  Sort: new  |  Time filter: all

✔ Page 1: fetched 4 posts  (running total: 4)
ℹ️  Reached last page.
Total posts collected: 4


### Results Preview 

In [19]:
# First 5 rows
df.head()

,title,selftext,created,ups,subreddit,url
0,Bulakenyong Tagalog,Pansin ko ang NCR iba ang gamit ng Tagalog mad...,2025-12-19 14:35:32 UTC,42,Tagalog,https://reddit.com/r/Tagalog/comments/1pqe7vl/...
1,When Do I Know If I Need My Wisdom Tooth Checked,This is how they look. Yung left yung medyo bo...,2025-08-24 11:44:01 UTC,1,DentistPh,https://reddit.com/r/DentistPh/comments/1mylfo...
2,nawawalan na ako ng pag asa,"sa 3 days before exam need ko mag aral, pero l...",2024-08-04 16:02:38 UTC,9,MedTechPH,https://reddit.com/r/MedTechPH/comments/1ejq3i...
3,"Got bitten by a dog, my sister makes it all ab...","I 27F, got bitten last night by one of our dog...",2024-07-17 23:19:14 UTC,119,OffMyChestPH,https://reddit.com/r/OffMyChestPH/comments/1e5...


In [20]:
# Upvote distribution
df["ups"].describe()

count      4.000000
mean      42.750000
std       53.841589
min        1.000000
25%        7.000000
50%       25.500000
75%       61.250000
max      119.000000
Name: ups, dtype: float64

In [21]:
# Top 10 posts by upvotes
df.sort_values("ups", ascending=False)[["title", "subreddit", "ups", "created"]].head(10)

,title,subreddit,ups,created
3,"Got bitten by a dog, my sister makes it all ab...",OffMyChestPH,119,2024-07-17 23:19:14 UTC
0,Bulakenyong Tagalog,Tagalog,42,2025-12-19 14:35:32 UTC
2,nawawalan na ako ng pag asa,MedTechPH,9,2024-08-04 16:02:38 UTC
1,When Do I Know If I Need My Wisdom Tooth Checked,DentistPh,1,2025-08-24 11:44:01 UTC


In [22]:
# Post count by subreddit
df["subreddit"].value_counts().head(10)

subreddit
Tagalog         1
DentistPh       1
MedTechPH       1
OffMyChestPH    1
Name: count, dtype: int64

### Save to CSV

In [23]:
save_results(df, OUTPUT_FILE)

💾 Appended to /Users/angelodelapaz/Documents/GitHub/healthphpersonal/data/raw/reddit/reddit_results.csv  (+4 posts, skipped 0 duplicates)
📊 Total rows in file: 3952


In [24]:
df.sample(10)

ValueError: Cannot take a larger sample than population when 'replace=False'